# 02 — Qualidade do onboarding: completude

**Dimensão:** completude (ver `AGENTS.md`, seção "Qualidade de dados").

**Objetivo:** medir quanto falta em cada coluna de `onboarding.csv` e separar o vazio opcional do vazio que é problema.

**Pedido:**
1. contar as linhas e os `user_id` únicos;
2. em cada coluna, contar os vazios: nulos e textos só com espaços, separados;
3. mostrar o percentual, da coluna mais vazia para a menos vazia;
4. dizer quais colunas o dicionário marca como opcionais.

**Referência (roteiro do professor):** 63.929 respostas, uma por conta; `objetivo_texto` com 5.467 vazios (8,6%), todos nulos, nenhum só com espaços; `nivel` (7,1%) e `dias_por_semana` (10,0%) também têm vazios sem serem opcionais.

Nada em `dados/brutos/` é alterado.

In [ ]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
# keep_default_na=False: só a célula vazia vira nulo. Sem isso, o pandas transformaria
# respostas como 'NA' ou 'None' em nulo e misturaria vazio de verdade com texto.
onboarding = pd.read_csv(BRUTOS / 'onboarding.csv', dtype=str, keep_default_na=False, na_values=[''])
onboarding.head()

## 1. Linhas e contas

In [ ]:
linhas = len(onboarding)
contas = onboarding['user_id'].nunique()
print(f'Linhas: {linhas:,} | user_id únicos: {contas:,}')
assert linhas == 63_929, 'dicionário: 63.929 linhas'
assert contas == linhas, 'esperado uma resposta por conta'

## 2 e 3. Vazios por coluna

Três tipos de vazio, contados separados:
- **nulo:** célula vazia no arquivo;
- **só espaços:** texto que, sem os espaços, fica vazio;
- **sem conteúdo:** texto preenchido que não informa nada (`nada`, `.`, `x`, `ok`...). Não está no pedido do professor, mas muda a conclusão para as colunas de texto livre.

A coluna **opcional** vem do dicionário (`docs/dicionario_dados.md`, seção 10): só `objetivo_texto` e `como_conheceu` são marcadas como opcionais.

In [ ]:
OPCIONAIS = {'objetivo_texto', 'como_conheceu'}  # dicionário, seção 10
SEM_CONTEUDO = {'nada', '😂', '.', 'x', 'sla', 'kkkk', 'asdf', 'ok', '-', 'nao sei', 'não sei'}

def vazios(coluna):
    s = onboarding[coluna]
    texto = s.dropna().str.strip().str.lower()
    return {
        'coluna': coluna,
        'nulos': s.isna().sum(),
        'so_espacos': (texto == '').sum(),
        'sem_conteudo': texto.isin(SEM_CONTEUDO).sum(),
        'opcional_no_dicionario': coluna in OPCIONAIS,
    }

completude = pd.DataFrame([vazios(c) for c in onboarding.columns]).set_index('coluna')
completude['vazios'] = completude['nulos'] + completude['so_espacos']
completude['% vazios'] = (completude['vazios'] / linhas * 100).round(1)
completude['% vazios + sem conteúdo'] = ((completude['vazios'] + completude['sem_conteudo']) / linhas * 100).round(1)
completude = completude.sort_values('% vazios', ascending=False)
completude

## Conferência com o roteiro

In [ ]:
obj = completude.loc['objetivo_texto']
assert obj['nulos'] == 5_467 and obj['so_espacos'] == 0, 'objetivo_texto: 5.467 nulos, nenhum só com espaços'
assert obj['% vazios'] == 8.6
assert completude.loc['nivel', '% vazios'] == 7.1
assert completude.loc['dias_por_semana', '% vazios'] == 10.0
print('Números do roteiro conferidos.')

## 4. Vazio opcional × vazio que é problema

In [ ]:
com_vazio = completude[completude['vazios'] > 0]
problema = com_vazio[~com_vazio['opcional_no_dicionario']]
print('Opcionais no dicionário, vazio esperado:', sorted(com_vazio[com_vazio['opcional_no_dicionario']].index))
print('Vazio sem ser opcional (problema):     ', sorted(problema.index))

## Respostas sem conteúdo

Como a lista foi montada: as respostas curtas mais repetidas. Conferimos abaixo que nenhuma resposta longa e frequente ficou de fora por engano, e mostramos exemplos.

In [ ]:
for col in ['objetivo_texto', 'como_conheceu']:
    texto = onboarding[col].dropna().str.strip().str.lower()
    print(f'--- {col}: 10 respostas mais repetidas')
    print(texto.value_counts().head(10).to_string(), '\n')

## Conclusão

- **63.929 respostas, uma por conta.** Bate com o dicionário.
- **Opcionais com vazio esperado:** `objetivo_texto` (8,6%) e `como_conheceu` (7,2%). Os vazios são todos nulos, nenhum só com espaços.
- **Vazio que é problema:** `nivel` (7,1%) e `dias_por_semana` (10,0%) têm vazios, e o dicionário não diz que são opcionais. Decisão: marcar como "não informado" e não imputar.
- **Além do roteiro:** `objetivo_texto` tem 13.768 respostas preenchidas sem conteúdo. Somadas aos nulos, cerca de **30% das contas não dizem o objetivo**. Para a extração com IA (objetivo 5), essas respostas não devem ir ao modelo: custam e não informam.
- **Próxima dimensão para este arquivo:** validade. `dias_por_semana` mistura `3`, `3x` e `3 vezes` (24.034 valores não numéricos).

Decisões registradas em `decisoes.md`.